<a href="https://colab.research.google.com/github/lucasferrara015/Portafolio-Data-Analysis/blob/main/Limpieza_y_transformaci%C3%B3n_Dataset_Netflix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# 1. Cargar el dataset
df = pd.read_csv('netflix_titles.csv')

# 2. Vista rápida
print("--- Primeras filas ---")
print(df.head(2))

# 3. Conteo de nulos ANTES
print("\n--- Conteo de Valores Nulos Inicial ---")
print(df.isnull().sum())

# 4. Tipos de datos iniciales
print("\n--- Información de Tipos de Datos ---")
print(df.info())

--- Primeras filas ---
  show_id     type                 title         director  \
0      s1    Movie  Dick Johnson Is Dead  Kirsten Johnson   
1      s2  TV Show         Blood & Water              NaN   

                                                cast        country  \
0                                                NaN  United States   
1  Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...   South Africa   

           date_added  release_year rating   duration  \
0  September 25, 2021          2020  PG-13     90 min   
1  September 24, 2021          2021  TV-MA  2 Seasons   

                                         listed_in  \
0                                    Documentaries   
1  International TV Shows, TV Dramas, TV Mysteries   

                                         description  
0  As her father nears the end of his life, filmm...  
1  After crossing paths at a party, a Cape Town t...  

--- Conteo de Valores Nulos Inicial ---
show_id            0
type             

In [ ]:
# Imputación de texto con un valor fijo
df['director'] = df['director'].fillna('Unknown')
df['cast'] = df['cast'].fillna('Unknown')
df['country'] = df['country'].fillna('Unknown')

# Imputación de rating con la moda
moda_rating = df['rating'].mode()[0]
df['rating'] = df['rating'].fillna(moda_rating)

# Eliminamos filas donde 'date_added' es nulo
df = df.dropna(subset=['date_added', 'duration'])

In [ ]:
# Limpiar espacios en blanco y convertir a datetime
df['date_added'] = df['date_added'].str.strip()
df['date_added'] = pd.to_datetime(df['date_added'], format='%B %d, %Y', errors='coerce')

# Feature Engineering: Extraer año y mes
df['year_added'] = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month_name()

# Categorizar el tipo de contenido (Movie / TV Show) para ahorrar memoria
df['type'] = df['type'].astype('category')

In [ ]:
# Separar texto: Nos quedamos solo con el primer género listado como "Género Principal"
df['primary_genre'] = df['listed_in'].str.split(',').str[0]

# Eliminación de columnas irrelevantes
# JUSTIFICACIÓN: 'show_id' es un índice redundante. 'description' requiere NLP avanzado.
columnas_a_eliminar = ['show_id', 'description']
df = df.drop(columns=columnas_a_eliminar)

In [ ]:
# 1. Extraer el número de la duración (solo para Películas)
df_movies = df[df['type'] == 'Movie'].copy()
df_movies['duration_num'] = df_movies['duration'].str.replace(' min', '').astype(float)

# 2. Calcular IQR
Q1 = df_movies['duration_num'].quantile(0.25)
Q3 = df_movies['duration_num'].quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

print(f"Límite inferior para películas: {limite_inferior} min")
print(f"Límite superior para películas: {limite_superior} min")

# 3. Identificar outliers
outliers = df_movies[(df_movies['duration_num'] < limite_inferior) | (df_movies['duration_num'] > limite_superior)]
print(f"Número de películas atípicas (muy cortas o muy largas): {len(outliers)}")

Límite inferior para películas: 46.5 min
Límite superior para películas: 154.5 min
Número de películas atípicas (muy cortas o muy largas): 450


In [ ]:
print("--- Conteo de Valores Nulos DESPUÉS ---")
print(df.isnull().sum())

# Exportar a CSV listo para análisis
df.to_csv('netflix_titles_clean.csv', index=False)
print("\n¡Dataset limpio exportado con éxito como 'netflix_titles_clean.csv'!")

--- Conteo de Valores Nulos DESPUÉS ---
type             0
title            0
director         0
cast             0
country          0
date_added       0
release_year     0
rating           0
duration         0
listed_in        0
year_added       0
month_added      0
primary_genre    0
dtype: int64

¡Dataset limpio exportado con éxito como 'netflix_titles_clean.csv'!
